# Cross-System Latency Analysis

Correlates **poc-deepgram** ground-truth audio timestamps with **Genesys Cloud** transcription delivery times to measure true end-to-end transcription pipeline latency.

**Ground truth**: poc-deepgram captures call audio independently, providing wall-clock timestamps for when speech actually occurred.

**Measurement point**: notifications-spike records `receivedAt` when Genesys transcription events arrive via WebSocket.

**Formula**:
```
true_latency = genesys_receivedAt - deepgram_audio_wall_clock_end
```

See `docs/cross_system_latency_plan.md` for the full plan.

---
## Module 1: Setup & Configuration

In [ ]:
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_style("whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

# Add parent directory to path so we can import scripts.correlate_latency
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from scripts.correlate_latency import (
    correlate,
    load_deepgram_session,
    load_genesys_conversation,
    match_utterances,
    CorrelationResult,
)

# === CONFIGURE THESE PATHS ===
# Point to your poc-deepgram session file and the matching Genesys conversation JSONL
DEEPGRAM_SESSION = REPO_ROOT / ".." / "poc-deepgram" / "results" / "REPLACE_ME.json"
GENESYS_CONVERSATION = REPO_ROOT / "conversation_events" / "REPLACE_ME.jsonl"

# Or auto-detect the most recent files
DEEPGRAM_RESULTS_DIR = (REPO_ROOT / ".." / "poc-deepgram" / "results").resolve()
GENESYS_EVENTS_DIR = (REPO_ROOT / "conversation_events").resolve()

OUTPUT_DIR = REPO_ROOT / "analysis_results" / "cross_system"
SAVE_DPI = 300
SIMILARITY_THRESHOLD = 0.55

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")

---
## Module 2: Auto-Detect & Load Data

Lists available sessions from both systems and lets you pick which to correlate.

In [ ]:
# List available Deepgram sessions
dg_files = sorted(DEEPGRAM_RESULTS_DIR.glob("*.json"), key=lambda f: f.stat().st_mtime, reverse=True)
print(f"Available Deepgram sessions ({len(dg_files)}):")
for i, f in enumerate(dg_files[:10]):
    data = json.loads(f.read_text())
    session = data.get("session", {})
    print(f"  [{i}] {f.name}  ({session.get('started_at', '?')} — {session.get('duration_seconds', '?')}s)")

print()

# List available Genesys conversations
gn_files = sorted(GENESYS_EVENTS_DIR.glob("*.jsonl"), key=lambda f: f.stat().st_mtime, reverse=True)
print(f"Available Genesys conversations ({len(gn_files)}):")
for i, f in enumerate(gn_files[:10]):
    lines = f.read_text().strip().splitlines()
    if lines:
        first = json.loads(lines[0])
        ts = datetime.fromtimestamp(first["receivedAt"], tz=timezone.utc)
        print(f"  [{i}] {f.name}  ({ts:%Y-%m-%d %H:%M} — {len(lines)} events)")
    else:
        print(f"  [{i}] {f.name}  (empty)")

In [ ]:
# === SELECT FILES ===
# Change these indices to select the desired session pair from the lists above
DG_INDEX = 0  # Index into dg_files list
GN_INDEX = 0  # Index into gn_files list

deepgram_path = dg_files[DG_INDEX]
genesys_path = gn_files[GN_INDEX]

print(f"Deepgram session: {deepgram_path.name}")
print(f"Genesys conversation: {genesys_path.name}")

---
## Module 3: Correlation & Matching

In [ ]:
# Run correlation
results = correlate(deepgram_path, genesys_path, similarity_threshold=SIMILARITY_THRESHOLD)

print(f"Matched utterance pairs: {len(results)}")

if not results:
    print("\nNo matches found. Check that:")
    print("  1. Both files cover the same call")
    print("  2. poc-deepgram captured the same audio Genesys transcribed")
    print("  3. The similarity threshold isn't too high")
else:
    # Build DataFrame for analysis
    df = pd.DataFrame([
        {
            "deepgram_transcript": r.deepgram_transcript,
            "genesys_transcript": r.genesys_transcript,
            "audio_wall_clock_end": r.audio_wall_clock_end,
            "genesys_received_at": r.genesys_received_at,
            "true_latency_s": r.true_latency_s,
            "true_latency_ms": r.true_latency_ms,
            "channel": r.channel,
            "similarity": r.similarity,
        }
        for r in results
    ])
    df["received_dt"] = pd.to_datetime(df["genesys_received_at"], unit="s", utc=True)
    df["audio_dt"] = pd.to_datetime(df["audio_wall_clock_end"], unit="s", utc=True)
    
    print(f"\nLatency range: {df['true_latency_ms'].min():.0f}ms — {df['true_latency_ms'].max():.0f}ms")
    print(f"Mean similarity: {df['similarity'].mean():.3f}")
    df.head()

---
## Module 4: Summary Statistics

In [ ]:
if results:
    latencies_ms = df["true_latency_ms"]
    latencies_s = df["true_latency_s"]
    
    print("=" * 60)
    print("CROSS-SYSTEM LATENCY SUMMARY")
    print("=" * 60)
    print(f"  Matched pairs:    {len(df)}")
    print(f"  Mean latency:     {latencies_ms.mean():.0f} ms ({latencies_s.mean():.3f} s)")
    print(f"  Median latency:   {latencies_ms.median():.0f} ms")
    print(f"  Std deviation:    {latencies_ms.std():.0f} ms")
    print(f"  Min latency:      {latencies_ms.min():.0f} ms")
    print(f"  Max latency:      {latencies_ms.max():.0f} ms")
    print()
    
    for p in [0.50, 0.75, 0.90, 0.95, 0.99]:
        print(f"  p{int(p*100):02d}:             {latencies_ms.quantile(p):.0f} ms")
    
    # By channel
    channels = df["channel"].unique()
    if len(channels) > 1:
        print(f"\nBy Channel:")
        for ch in sorted(channels):
            ch_data = df[df["channel"] == ch]["true_latency_ms"]
            print(f"  {ch}: median={ch_data.median():.0f}ms, mean={ch_data.mean():.0f}ms, n={len(ch_data)}")

---
## Module 5: Visualizations

### Chart 1: True Latency Distribution

In [ ]:
if results:
    fig, ax = plt.subplots(figsize=(12, 5))
    
    ax.hist(df["true_latency_ms"], bins=30, edgecolor="black", alpha=0.7, color="steelblue")
    ax.axvline(df["true_latency_ms"].mean(), color="red", linestyle="--", 
               label=f"Mean: {df['true_latency_ms'].mean():.0f}ms")
    ax.axvline(df["true_latency_ms"].median(), color="green", linestyle="--",
               label=f"Median: {df['true_latency_ms'].median():.0f}ms")
    ax.set_xlabel("True Latency (ms)")
    ax.set_ylabel("Count")
    ax.set_title("Cross-System Transcription Latency Distribution", fontsize=14, fontweight="bold")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "cross_latency_distribution.png", dpi=SAVE_DPI, bbox_inches="tight")
    plt.show()

### Chart 2: Latency Timeline (scatter + trend)

In [ ]:
if results and len(df) > 1:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for channel, color in [("INTERNAL", "steelblue"), ("EXTERNAL", "darkorange")]:
        mask = df["channel"] == channel
        if mask.any():
            subset = df[mask].sort_values("audio_dt")
            ax.scatter(subset["audio_dt"], subset["true_latency_ms"],
                       alpha=0.6, s=40, color=color, label=channel)
    
    # Rolling average trend
    df_sorted = df.sort_values("audio_dt")
    window = max(3, len(df_sorted) // 10)
    rolling = df_sorted["true_latency_ms"].rolling(window=window, min_periods=1).mean()
    ax.plot(df_sorted["audio_dt"].values, rolling.values, color="red", linewidth=2,
            alpha=0.8, label=f"Rolling avg (w={window})")
    
    ax.set_xlabel("Audio Time (UTC)")
    ax.set_ylabel("True Latency (ms)")
    ax.set_title("Cross-System Latency Over Time", fontsize=14, fontweight="bold")
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "cross_latency_timeline.png", dpi=SAVE_DPI, bbox_inches="tight")
    plt.show()

### Chart 3: Latency by Channel (box plot)

In [ ]:
if results and len(df["channel"].unique()) > 1:
    fig, ax = plt.subplots(figsize=(8, 6))
    
    sns.boxplot(
        data=df, x="channel", y="true_latency_ms",
        palette={"INTERNAL": "steelblue", "EXTERNAL": "darkorange"},
        ax=ax, showfliers=True,
        flierprops={"marker": ".", "alpha": 0.3},
    )
    ax.set_xlabel("Channel")
    ax.set_ylabel("True Latency (ms)")
    ax.set_title("Cross-System Latency by Channel", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "cross_latency_boxplot.png", dpi=SAVE_DPI, bbox_inches="tight")
    plt.show()

### Chart 4: Match Quality — Similarity vs Latency

In [ ]:
if results:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    scatter = ax.scatter(
        df["similarity"], df["true_latency_ms"],
        c=df["channel"].map({"INTERNAL": "steelblue", "EXTERNAL": "darkorange"}).fillna("gray"),
        alpha=0.6, s=50, edgecolors="black", linewidth=0.5,
    )
    ax.set_xlabel("Text Similarity Score")
    ax.set_ylabel("True Latency (ms)")
    ax.set_title("Match Quality vs Latency", fontsize=14, fontweight="bold")
    
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="steelblue", markersize=8, label="INTERNAL"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="darkorange", markersize=8, label="EXTERNAL"),
    ]
    ax.legend(handles=legend_elements)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "cross_latency_match_quality.png", dpi=SAVE_DPI, bbox_inches="tight")
    plt.show()

---
## Module 6: Comparison with Genesys Self-Reported Latency

Compare the cross-system (ground truth) latency with the Genesys-only estimated latency from the existing analysis.

In [ ]:
# Load the Genesys-only latency summary if available
genesys_summary_path = REPO_ROOT / "analysis_results" / "latency_summary.json"

if results and genesys_summary_path.exists():
    genesys_summary = json.loads(genesys_summary_path.read_text())
    gs = genesys_summary["overall"]
    
    cross_median = df["true_latency_s"].median()
    cross_p95 = df["true_latency_s"].quantile(0.95)
    
    print("=" * 60)
    print("COMPARISON: Cross-System vs Genesys Self-Reported")
    print("=" * 60)
    print(f"{'Metric':<20} {'Cross-System':>15} {'Genesys-Only':>15} {'Delta':>10}")
    print("-" * 60)
    
    for label, cross_val, genesys_val in [
        ("Median (p50)", cross_median, gs["p50"]),
        ("p95", cross_p95, gs["p95"]),
        ("Mean", df["true_latency_s"].mean(), gs["mean"]),
    ]:
        delta = cross_val - genesys_val
        print(f"{label:<20} {cross_val:>13.3f}s {genesys_val:>13.3f}s {delta:>+9.3f}s")
    
    print(f"\nNote: Positive delta means cross-system measures HIGHER latency")
    print(f"(expected, since Genesys self-reported latency underestimates due to anchor bias)")
elif not genesys_summary_path.exists():
    print("Genesys-only summary not found. Run the latency_analysis notebook first.")

---
## Module 7: Matched Pairs Detail Table

In [ ]:
if results:
    print(f"{'Latency':>10}  {'Sim':>5}  {'Ch':>8}  {'Deepgram Transcript':<40}  {'Genesys Transcript':<40}")
    print("-" * 120)
    for _, row in df.iterrows():
        dg_trunc = row["deepgram_transcript"][:38] + (".." if len(row["deepgram_transcript"]) > 38 else "")
        gn_trunc = row["genesys_transcript"][:38] + (".." if len(row["genesys_transcript"]) > 38 else "")
        print(f"{row['true_latency_ms']:>8.0f}ms  {row['similarity']:>5.2f}  {row['channel']:>8}  {dg_trunc:<40}  {gn_trunc:<40}")

---
## Module 8: Export Results

In [ ]:
if results:
    from scripts.correlate_latency import export_csv
    
    csv_path = OUTPUT_DIR / "correlation_results.csv"
    export_csv(results, csv_path)
    
    # Also save summary JSON
    summary = {
        "deepgram_session": str(deepgram_path.name),
        "genesys_conversation": str(genesys_path.name),
        "matched_pairs": len(results),
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "latency_ms": {
            "mean": round(df["true_latency_ms"].mean(), 1),
            "median": round(df["true_latency_ms"].median(), 1),
            "std": round(df["true_latency_ms"].std(), 1),
            "min": round(df["true_latency_ms"].min(), 1),
            "max": round(df["true_latency_ms"].max(), 1),
            "p50": round(df["true_latency_ms"].quantile(0.50), 1),
            "p95": round(df["true_latency_ms"].quantile(0.95), 1),
            "p99": round(df["true_latency_ms"].quantile(0.99), 1),
        },
        "mean_similarity": round(df["similarity"].mean(), 3),
    }
    
    summary_path = OUTPUT_DIR / "correlation_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2))
    
    print(f"\nExported:")
    print(f"  CSV: {csv_path}")
    print(f"  JSON: {summary_path}")